<a href="https://colab.research.google.com/github/Joan2022Laurente/fade/blob/main/notebooks/GLOBALent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [ ]:
# ================================================================
#   SETUP UNIFICADO — Z-Image Turbo + Qwen Image Edit AIO
#   Dark Beast · Qwen3-4B · Qwen2.5-VL · VAE · LoRA · FaceMask
#   Google Colab · GPU ≥15 GB VRAM
#   ▸ Todo se guarda en /content — NO usa Google Drive
#   ▸ Ejecutar UNA SOLA VEZ, luego reiniciar y lanzar ComfyUI
# ================================================================

# ── CHECKPOINTS (deja URL vacía para omitir) ───────────────────
CKPT1_URL  = "https://civitai.red/api/download/models/2957651?fileId=2837007"  # @param {type:"string"}
CKPT1_NAME = "divingZimageTurbo.safetensors"                                   # @param {type:"string"}
CKPT2_URL  = ""                                                                 # @param {type:"string"}
CKPT2_NAME = "darkBeast.safetensors"                                            # @param {type:"string"}
CKPT3_URL  = ""                                                                 # @param {type:"string"}
CKPT3_NAME = "ZiT.safetensors"                                                  # @param {type:"string"}
CKPT4_URL  = ""                                                                 # @param {type:"string"}
CKPT4_NAME = "byStable_Yogi.safetensors"                                        # @param {type:"string"}

# ── LoRAs (deja URL vacía para omitir) ────────────────────────
LORA_HF_ENABLED = True                                                          # @param {type:"boolean"}
LORA1_URL  = ""                                                                 # @param {type:"string"}
LORA1_NAME = "lora1.safetensors"                                                # @param {type:"string"}
LORA2_URL  = ""  # @param {type:"string"}
LORA2_NAME = "nsMASTER.safetensors"                                             # @param {type:"string"}
LORA3_URL  = ""                                                                 # @param {type:"string"}
LORA3_NAME = "lora3.safetensors"                                                # @param {type:"string"}
LORA4_URL  = ""  # @param {type:"string"}
LORA4_NAME = "pussyOnly.safetensors"                                            # @param {type:"string"}

# ══════════════════════════════════════════════════════════════
import os, subprocess, sys, shutil, concurrent.futures
from google.colab import userdata

CIVITAI_KEY   = userdata.get("CIVITAI_KEY")
TAILSCALE_KEY = userdata.get("TAILSCALE_KEY")
HF_TOKEN      = userdata.get("HF_TOKEN")

missing = [k for k, v in [("CIVITAI_KEY", CIVITAI_KEY),
                           ("TAILSCALE_KEY", TAILSCALE_KEY),
                           ("HF_TOKEN", HF_TOKEN)] if not v]
if missing:
    raise RuntimeError(f"❌ Faltan secrets: {', '.join(missing)}")
print("✅ Secrets cargados")

# ── RUTAS GLOBALES ─────────────────────────────────────────────
COMFY          = "/content/ComfyUI"
MODELS         = f"{COMFY}/models"
CUSTOM_NODES   = f"{COMFY}/custom_nodes"
WORKFLOWS      = f"{COMFY}/user/default/workflows"

# ── URLs base HuggingFace ─────────────────────────────────────
HF_Z         = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files"
HF_LORA_BASE = "https://huggingface.co/joanjlau/loraprueba0/resolve/main"
HF_QWEN_BASE = "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main"

# ── Listas internas ───────────────────────────────────────────
CHECKPOINTS   = [(CKPT1_URL, CKPT1_NAME), (CKPT2_URL, CKPT2_NAME),
                 (CKPT3_URL, CKPT3_NAME), (CKPT4_URL, CKPT4_NAME)]
LORAS_CIVITAI = [(LORA1_URL, LORA1_NAME), (LORA2_URL, LORA2_NAME),
                 (LORA3_URL, LORA3_NAME), (LORA4_URL, LORA4_NAME)]

# ══════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════
def run(cmd, cwd=None, check=False):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout.strip():
        print(r.stdout.strip()[-2000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"Falló: {cmd}")
    return r.returncode

def pip_install(*pkgs):
    for pkg in pkgs:
        run(f'"{sys.executable}" -m pip install {pkg} -q --no-deps 2>/dev/null || '
            f'"{sys.executable}" -m pip install {pkg} -q')

def clone(url, dst, name=None):
    label = name or dst.split("/")[-1]
    if os.path.isdir(dst):
        print(f"  ↻  {label} ya existe — actualizando")
        run("git pull --ff-only", cwd=dst)
    else:
        run(f'git clone --depth 1 "{url}" "{dst}"')
    req = f"{dst}/requirements.txt"
    if os.path.exists(req):
        run(f'"{sys.executable}" -m pip install -r "{req}" -q')

def file_ok(path):
    return os.path.exists(path) and os.path.getsize(path) > 1_048_576

def dl(url, folder, name, token_header=None):
    os.makedirs(folder, exist_ok=True)
    dest = f"{folder}/{name}"
    if file_ok(dest):
        print(f"  ✅ {name} ya existe — skip")
        return dest
    print(f"  ↓ {name}")
    auth = f'--header="Authorization: Bearer {token_header}"' if token_header else ""
    # Intenta aria2c primero, wget como fallback
    ret = run(
        f'aria2c -c -x 16 -s 16 -k 1M --file-allocation=none '
        f'--console-log-level=error --summary-interval=0 '
        f'{auth} -d "{folder}" -o "{name}" "{url}"'
    )
    if ret != 0 or not file_ok(dest):
        print(f"  → aria2c falló, intentando wget...")
        auth_wget = f'--header="Authorization: Bearer {token_header}"' if token_header else ""
        run(f'wget -q --show-progress {auth_wget} -O "{dest}" "{url}"')
    ok = file_ok(dest)
    if ok:
        print(f"  ✅ {name}  ({os.path.getsize(dest)/1024**3:.2f} GB)")
    else:
        print(f"  ❌ {name}")
    return dest if ok else None

def dl_hf(url, folder, name):
    return dl(url, folder, name, token_header=HF_TOKEN)

def dl_civitai(url, folder, name):
    sep = "&" if "?" in url else "?"
    return dl(f"{url}{sep}token={CIVITAI_KEY}", folder, name)

def dl_parallel(tasks, workers=4):
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:
        fs = {ex.submit(fn, url, folder, name): name
              for fn, url, folder, name in tasks}
        for f in concurrent.futures.as_completed(fs):
            try:
                f.result()
            except Exception as e:
                print(f"  ❌ Error: {e}")


def verify(label, path, is_dir=False):
    ok = os.path.isdir(path) if is_dir else file_ok(path)
    print(f"  {'✅' if ok else '❌'} {label}")

# ══════════════════════════════════════════════════════════════
print("\n[SYS] Herramientas del sistema...")
run("apt-get update -qq && apt-get install -y -qq aria2 git wget curl", check=True)

# ══════════════════════════════════════════════════════════════
print("\n[1] ComfyUI...")
clone("https://github.com/comfyanonymous/ComfyUI.git", COMFY, "ComfyUI")
run(f'"{sys.executable}" -m pip install -r "{COMFY}/requirements.txt" -q')

for d in [
    f"{MODELS}/diffusion_models",
    f"{MODELS}/text_encoders",
    f"{MODELS}/vae",
    f"{MODELS}/loras",
    f"{MODELS}/loras/z-image",
    f"{MODELS}/upscale_models",
    f"{MODELS}/unet",
    f"{MODELS}/checkpoints",
    f"{MODELS}/clip",
    CUSTOM_NODES,
    WORKFLOWS,
]:
    os.makedirs(d, exist_ok=True)

# ══════════════════════════════════════════════════════════════
print("\n[2] Custom nodes...")

# ── Entorno 1 ─────────────────────────────────────────────────
clone("https://github.com/rgthree/rgthree-comfy.git",
      f"{CUSTOM_NODES}/rgthree-comfy",               "rgthree-comfy")
clone("https://github.com/Joan2022Laurente/node.git",
      f"{CUSTOM_NODES}/azzia-nodes",                  "azzia-nodes")
clone("https://github.com/kijai/ComfyUI-KJNodes.git",
      f"{CUSTOM_NODES}/causachuperi",                 "ComfyUI-KJNodes")
clone("https://github.com/city96/ComfyUI-GGUF.git",
      f"{CUSTOM_NODES}/ComfyUI-GGUF",                 "ComfyUI-GGUF")
clone("https://github.com/cubiq/ComfyUI_essentials.git",
      f"{CUSTOM_NODES}/ComfyUI_essentials",            "ComfyUI_essentials")
clone("https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git",
      f"{CUSTOM_NODES}/ComfyUI_UltimateSDUpscale",     "ComfyUI_UltimateSDUpscale")
clone("https://github.com/chrisgoringe/cg-use-everywhere.git",
      f"{CUSTOM_NODES}/cg-use-everywhere",             "cg-use-everywhere")
clone("https://github.com/alexopus/ComfyUI-Image-Saver.git",
      f"{CUSTOM_NODES}/ComfyUI-Image-Saver",           "ComfyUI-Image-Saver")
clone("https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
      f"{CUSTOM_NODES}/ComfyUI-Impact-Pack",           "ComfyUI-Impact-Pack")
clone("https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git",
      f"{CUSTOM_NODES}/ComfyUI-Impact-Subpack",        "ComfyUI-Impact-Subpack")

# ── Entorno 2 (nuevos, sin duplicar los anteriores) ───────────
clone("https://github.com/lrzjason/Comfyui-QwenEditUtils.git",
      f"{CUSTOM_NODES}/Comfyui-QwenEditUtils",         "Comfyui-QwenEditUtils")
clone("https://github.com/Fannovel16/comfyui_controlnet_aux.git",
      f"{CUSTOM_NODES}/comfyui_controlnet_aux",        "comfyui_controlnet_aux")
clone("https://github.com/MohammadAboulEla/ComfyUI-iTools.git",
      f"{CUSTOM_NODES}/ComfyUI-iTools",                "ComfyUI-iTools")

# ══════════════════════════════════════════════════════════════
print("\n[3] Dependencias Python...")
pip_install(
    # ── Base / Entorno 1
    "accelerate", "einops", "torchvision", "Pillow",
    # ── Impact Pack
    "ultralytics", "dlib", "segment-anything", "mediapipe",
    # ── Entorno 2
    "transformers>=4.45.0", "opencv-python-headless",
    "scipy", "scikit-image", "gguf",
    "timm", "controlnet-aux",
)
print("  ✅ Dependencias instaladas")

# ══════════════════════════════════════════════════════════════
print("\n[4] Modelo de detección facial (Impact Pack)...")
BBOX_DIR = f"{MODELS}/ultralytics/bbox"
os.makedirs(BBOX_DIR, exist_ok=True)
dl_hf(
    "https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt",
    BBOX_DIR,
    "face_yolov8m.pt",
)

# ══════════════════════════════════════════════════════════════
print("\n[5] Descarga paralela de modelos...")
tasks = [
    # ── Text encoders — Z-Image Turbo ─────────────────────────
    (dl_hf,
     "https://huggingface.co/mradermacher/Qwen3-4B-heretic-GGUF/resolve/main/Qwen3-4B-heretic.Q6_K.gguf",
     f"{MODELS}/text_encoders", "qwen3-4b-heretic-Q6_K.gguf"),

    # ── Text encoder — Qwen Image Edit AIO ───────────────────
    (dl_hf,
     f"{HF_QWEN_BASE}/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors",
     f"{MODELS}/text_encoders", "qwen_2.5_vl_7b_fp8_scaled.safetensors"),

    # ── VAE ───────────────────────────────────────────────────
    (dl_hf,
     f"{HF_Z}/vae/ae.safetensors",
     f"{MODELS}/vae", "ae.safetensors"),
    (dl_hf,
     f"{HF_QWEN_BASE}/split_files/vae/qwen_image_vae.safetensors",
     f"{MODELS}/vae", "qwen_image_vae.safetensors"),

    # ── Upscale model ─────────────────────────────────────────
    (dl_hf,
     "https://huggingface.co/Phips/4xFaceUpSharpDAT/resolve/main/4xFaceUpSharpDAT.safetensors",
     f"{MODELS}/upscale_models", "4xFaceUpSharpDAT.safetensors"),

    # ── Modelo GGUF Qwen Image Edit ───────────────────────────
    (dl_hf,
     "https://huggingface.co/Novice25/Qwen-Image-Edit-Rapid-AIO-GGUF/resolve/main/v23/Qwen-Rapid-NSFW-v23_Q4_K.gguf",
     f"{MODELS}/unet", "Qwen-Rapid-NSFW-v23_Q4_K.gguf"),
]

# ── Checkpoints Civitai ───────────────────────────────────────
for url, name in CHECKPOINTS:
    if url.strip():
        tasks.append((dl_civitai, url.strip(), f"{MODELS}/diffusion_models", name.strip()))
    else:
        print(f"  ⚪ {name or 'Checkpoint vacío'}: skip")

# ── LoRAs HF ─────────────────────────────────────────────────
if LORA_HF_ENABLED:
    tasks.append((dl_hf,
                  f"{HF_LORA_BASE}/YummyHDzit.safetensors",
                  f"{MODELS}/loras", "YummyHDzit.safetensors"))
    tasks.append((dl_hf,
                  f"{HF_LORA_BASE}/yummy_kitty.safetensors",
                  f"{MODELS}/loras/z-image", "yummy_kitty.safetensors"))
else:
    print("  ⚪ LoRAs HF: desactivadas")

# ── LoRAs Civitai ─────────────────────────────────────────────
for url, name in LORAS_CIVITAI:
    if url.strip():
        tasks.append((dl_civitai, url.strip(), f"{MODELS}/loras", name.strip()))
    else:
        print(f"  ⚪ {name or 'LoRA vacío'}: skip")

dl_parallel(tasks, workers=4)

# ══════════════════════════════════════════════════════════════
print("\n[6] Workflows...")
WF_SRC_DIR = f"{CUSTOM_NODES}/azzia-nodes/workflows"
if os.path.isdir(WF_SRC_DIR):
    json_files = sorted(f for f in os.listdir(WF_SRC_DIR) if f.endswith(".json"))
    if json_files:
        for wf in json_files:
            shutil.copy2(f"{WF_SRC_DIR}/{wf}", f"{WORKFLOWS}/{wf}")
            print(f"  ✅ {wf}")
        print(f"  → {len(json_files)} workflow(s) copiados desde azzia-nodes/workflows/")
    else:
        print("  ⚠️  azzia-nodes/workflows/ existe pero está vacía")
else:
    print("  ⚠️  azzia-nodes/workflows/ no encontrada — ¿el repo ya tiene esa carpeta?")

# ══════════════════════════════════════════════════════════════
print("\n" + "═"*60)
print("   VERIFICACIÓN FINAL")
print("═"*60)

print("\n  — Text Encoders —")
verify("Qwen3-4B-heretic Q6_K GGUF",
       f"{MODELS}/text_encoders/qwen3-4b-heretic-Q6_K.gguf")
verify("Qwen2.5-VL 7B FP8",
       f"{MODELS}/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors")

print("\n  — VAE —")
verify("Flux VAE (ae.safetensors)",
       f"{MODELS}/vae/ae.safetensors")
verify("Qwen Image VAE",
       f"{MODELS}/vae/qwen_image_vae.safetensors")

print("\n  — Modelos UNet / Diffusion —")
verify("Qwen-Rapid-NSFW-v23 Q4_K GGUF",
       f"{MODELS}/unet/Qwen-Rapid-NSFW-v23_Q4_K.gguf")

print("\n  — Upscale —")
verify("4xFaceUpSharpDAT",
       f"{MODELS}/upscale_models/4xFaceUpSharpDAT.safetensors")

print("\n  — Detección facial —")
verify("face_yolov8m.pt",
       f"{MODELS}/ultralytics/bbox/face_yolov8m.pt")

print("\n  — Custom nodes —")
for label, folder in [
    ("rgthree-comfy",               "rgthree-comfy"),
    ("azzia-nodes",                 "azzia-nodes"),
    ("ComfyUI-GGUF",                "ComfyUI-GGUF"),
    ("ComfyUI_essentials",          "ComfyUI_essentials"),
    ("ComfyUI_UltimateSDUpscale",   "ComfyUI_UltimateSDUpscale"),
    ("cg-use-everywhere",           "cg-use-everywhere"),
    ("ComfyUI-Image-Saver",         "ComfyUI-Image-Saver"),
    ("ComfyUI-Impact-Pack",         "ComfyUI-Impact-Pack"),
    ("ComfyUI-Impact-Subpack",      "ComfyUI-Impact-Subpack"),
    ("Comfyui-QwenEditUtils",       "Comfyui-QwenEditUtils"),
    ("comfyui_controlnet_aux",      "comfyui_controlnet_aux"),
    ("ComfyUI-iTools",              "ComfyUI-iTools"),
]:
    verify(label, f"{CUSTOM_NODES}/{folder}", is_dir=True)

print("\n  — Checkpoints —")
for url, name in CHECKPOINTS:
    if url.strip():
        verify(name, f"{MODELS}/diffusion_models/{name}")
    else:
        print(f"  ⚪ {name}: no configurado")

print("\n  — LoRAs —")
if LORA_HF_ENABLED:
    verify("YummyHDzit.safetensors      → loras/",
           f"{MODELS}/loras/YummyHDzit.safetensors")
    verify("yummy_kitty.safetensors     → loras/z-image/",
           f"{MODELS}/loras/z-image/yummy_kitty.safetensors")
else:
    print("  ⚪ LoRAs HF: desactivadas")
for url, name in LORAS_CIVITAI:
    if url.strip():
        verify(name, f"{MODELS}/loras/{name}")
    else:
        print(f"  ⚪ {name}: no configurado")

print("\n" + "═"*60)
print("✅ SETUP COMPLETO")
print()
print("→ Z-Image Turbo UNet (descarga por separado si no lo tienes):")
print("    z_image_turbo_bf16.safetensors → models/diffusion_models/")
print()
print("→ KSampler Z-Image : Steps=8-10 · CFG=1.0 · euler · simple")
print("→ CLIP Z-Image     : qwen3-4b-heretic-Q6_K.gguf  |  VAE: ae.safetensors")
print("→ Qwen Edit        : Qwen-Rapid-NSFW-v23_Q4_K.gguf | VAE: qwen_image_vae.safetensors")

   SETUP — Qwen Image Edit AIO  ·  Google Colab GGUF v23 Q4_K

[SYS] Instalando herramientas del sistema...
  $ apt-get update -qq && apt-get install -y -qq aria2 git wget curl
2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Preparing to unpack .../3-libcurl4-openssl-dev_7.81.0-1ubuntu1.24_amd64.deb ...
Unpacking libcurl4-openssl-dev:amd64 (7.81.0-1ubuntu1.24) over (7.81.0-1ubuntu1.23) ...
Preparing to unpack .../4-curl_7.81.0-1ubuntu1.24_amd64.deb ...
Unpacking curl (7.81.0-1ubuntu1.24) over (7.81.0-1ubuntu1.23) ...
Preparing to unpack .../5-libcurl4_7.81.0-1ubuntu1.24_amd64.deb ...
Unpacking libcurl4:amd64 (7.81.0-1ubuntu1.24) over (7.81.0-1ubuntu1.23) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libcurl4:amd64 (7.81.0-1ubuntu1.24) ...
Setting up curl (7.81.0-1ubuntu1.24) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Setting up libcurl4-openssl-dev:amd64 (7.81.0-1ubuntu1.24) ...
Processing triggers for man-db (2.10.

In [ ]:
# 🔒 Lanzar ComfyUI con Tailscale

import os, time
from google.colab import userdata

TAILSCALE_AUTH_KEY = userdata.get('TAILSCALE_KEY')
if not TAILSCALE_AUTH_KEY:
    raise RuntimeError("❌ Falta TAILSCALE_KEY en Secrets de Colab")

# 1. Instalar Tailscale
!curl -fsSL https://tailscale.com/install.sh | sh

# 2. Iniciar daemon
print("🚀 Iniciando tailscaled...")
!nohup tailscaled --tun=userspace-networking --socks5-server=localhost:1055 > /tmp/tailscaled.log 2>&1 &
time.sleep(5)

# 3. Verificar
print("🔍 Verificando tailscaled...")
!ps aux | grep tailscaled

# 4. Conectar
print("🔗 Conectando a Tailscale...")
!tailscale up --authkey={TAILSCALE_AUTH_KEY} --hostname=colab-comfyui

# 5. Mostrar IP
print("\n" + "="*50)
print("🌐 TU IP PRIVADA DE TAILSCALE ES:")
!tailscale ip -4
print("="*50)

# 6. Lanzar ComfyUI
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("\n🚀 Iniciando ComfyUI...")
%cd /content/ComfyUI
!python main.py --listen 0.0.0.0 --dont-print-server --lowvram


Installing Tailscale for ubuntu jammy, using method apt
+ mkdir -p --mode=0755 /usr/share/keyrings
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.noarmor.gpg
+ tee /usr/share/keyrings/tailscale-archive-keyring.gpg
+ chmod 0644 /usr/share/keyrings/tailscale-archive-keyring.gpg
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.tailscale-keyring.list
+ tee /etc/apt/sources.list.d/tailscale.list
# Tailscale packages for ubuntu jammy
deb [signed-by=/usr/share/keyrings/tailscale-archive-keyring.gpg] https://pkgs.tailscale.com/stable/ubuntu jammy main
+ chmod 0644 /etc/apt/sources.list.d/tailscale.list
+ apt-get update
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://clo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 🔒 Lanzar ComfyUI con Cloudflare Tunnel

import os
import subprocess
import threading
import time

# --- CONFIGURACIÓN E INSTALACIÓN ---

print("📦 Instalando Cloudflared...")
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# --- FUNCIÓN PARA EL TÚNEL ---

def run_cloudflare():
    # Creamos el túnel apuntando al puerto 8188 (por defecto de ComfyUI)
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    # Buscamos la URL generada en los logs
    for line in p.stdout:
        if "trycloudflare.com" in line:
            print("\n" + "="*60)
            print("🌐 TU URL PÚBLICA ES:")
            print(line.strip().split(" ")[-1])
            print("="*60 + "\n")
        # Si quieres ver los logs de cloudflare, descomenta la siguiente línea:
        # print(line, end="")

# 1. Iniciar el túnel en segundo plano
threading.Thread(target=run_cloudflare, daemon=True).start()

# 2. Esperar un momento a que el túnel se establezca
time.sleep(5)

# 3. Lanzar ComfyUI
print("🚀 Iniciando ComfyUI...")
%cd /content/ComfyUI
# Usamos 127.0.0.1 porque el túnel de Cloudflare está escuchando localmente
!python main.py --dont-print-server